# Experiment A — X/Y semantic distance and historical Probe performance
CPU only; no OpenAI API calls; no Drive. All 50 candidates are retained. Uniform definitions are newly drafted by the assistant and **not independently reviewed**; existing definition sets are analyzed separately. Historical v5 uses 8 train/8 eval and 499 permutations, not v0.97 formal selection. L1–21 are unavailable. Run all cells; GitHub secret `GITHUB_RESULTS_TOKEN` is optional for local analysis, required for durable checkpoints. Never modify historical runs.


In [ ]:
%pip -q install sentence-transformers==3.4.1 huggingface-hub==0.29.3


In [ ]:
import os, io, json, hashlib, base64, platform, importlib.metadata as im
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from scipy.stats import pearsonr, spearmanr, rankdata
import matplotlib.pyplot as plt
from IPython.display import display
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from urllib.parse import quote
CFG = {"repo":"ryouy/GraduationThesis","source_main":"a33c4986656689969b79bda129f79d0d7bdbe2f5","source_cue":"c9f8d3d563f51d6bead9a3221dfa81554d23102d","branch":"experiments/concept-pair-selection-v1","experiment_dir":"experiments/2026-11/1126_concept_pair_selection/","run_id":"20261126_concept_pair_distance_A_v1","embedding":"sentence-transformers/all-mpnet-base-v2","revision":"e8c3b32edf5434bc2275fc9bab85f82640a19130","seed":42,"auc_min":0.8,"q_max":0.05,"bootstrap":2000,"primary_layer":24,"api_calls":0,"definitions_status":"assistant-authored provisional standardized definitions; no external generation API; separate existing-definition sensitivity analysis"}
# Optional: change run_id BEFORE running to keep a separate version.
SAVE_GITHUB = True
ROOT = Path('/content') / CFG['run_id']; ROOT.mkdir(exist_ok=True)
REPO = CFG['repo']; API = f'https://api.github.com/repos/{REPO}'
DEST = CFG['experiment_dir']+'results/'+CFG['run_id']+'/'
S = requests.Session()
try: token = userdata.get('GITHUB_RESULTS_TOKEN')
except Exception: token = None
if token: S.headers['Authorization'] = 'Bearer '+token
if SAVE_GITHUB and not token:
    print('No GitHub token: local analysis only; download ZIP at end.')
    SAVE_GITHUB = False
def digest(b): return hashlib.sha256(b).hexdigest()
def get_raw(ref,path):
    url=f'https://raw.githubusercontent.com/{REPO}/{ref}/{path}'
    response=requests.get(url,timeout=90); response.raise_for_status()
    return response.content
def get_remote(name):
    if not SAVE_GITHUB: return None
    resp=S.get(f'{API}/contents/{DEST}{name}',params={'ref':CFG['branch']},timeout=60)
    if resp.status_code==404: return None
    resp.raise_for_status()
    return base64.b64decode(resp.json()['content'])
def put_remote(name,data):
    if not SAVE_GITHUB: return
    prior=get_remote(name)
    if prior is not None:
        if prior!=data: raise RuntimeError(f'Existing output differs: {name}; change run_id, no overwrite.')
        return
    resp=S.put(f'{API}/contents/{DEST}{name}',json={'message':f'Experiment A: {name}','content':base64.b64encode(data).decode(),'branch':CFG['branch']},timeout=90)
    resp.raise_for_status()
def save(name,data,remote=False):
    if isinstance(data,str): data=data.encode()
    (ROOT/name).write_bytes(data)
    if remote: put_remote(name,data)
def save_df(name,df,remote=False):
    save(name,df.to_csv(index=False,lineterminator='\n'),remote)
print('Run:',CFG['run_id'],'GitHub checkpoint:',SAVE_GITHUB)


In [ ]:
P='experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/'
C='experiments/2026-11/1126_cue_selection_luna/results/20261126_cue_selection_luna_v1/'
V='experiments/2026-11/1126_cue_selection_luna/results/20261126_cue_y_optimization_v4/'
paths={
 'pairs':(CFG['source_main'],'docs/concept_pair_candidates_50_v0_1.csv'),
 'scores':(CFG['source_main'],P+'pair_probe_scores.csv'),
 'old_config':(CFG['source_main'],P+'config.json'),
 'existing_cue':(CFG['source_cue'],C+'definitions.csv'),
 'existing_pre':(CFG['source_main'],'experiments/2026-11/1124_pre_main_experiment/results/20261124_pre_main_v1/definitions.csv'),
 'cue_config':(CFG['source_cue'],C+'config.json'),
 'v4':(CFG['source_cue'],V+'summary.csv')}
raw={k:get_raw(*v) for k,v in paths.items()}
# Pin the authored definition file by content, not a moving branch.
DRAFT_CSV = "concept,definition,provenance\n\"bicycle\",\"A human-powered vehicle with two wheels, pedals and handlebars.\",\"draft_uniform_v1\"\n\"submarine\",\"A vessel designed to travel and operate underwater.\",\"draft_uniform_v1\"\n\"boat\",\"A watercraft designed to carry people or goods across water.\",\"draft_uniform_v1\"\n\"bus\",\"A road vehicle designed to transport many passengers.\",\"draft_uniform_v1\"\n\"helicopter\",\"An aircraft lifted and propelled by rotating overhead blades.\",\"draft_uniform_v1\"\n\"motorcycle\",\"A powered two-wheeled road vehicle ridden astride a seat.\",\"draft_uniform_v1\"\n\"elephant\",\"A large land mammal with a long trunk and broad ears.\",\"draft_uniform_v1\"\n\"lion\",\"A large social predatory cat whose males commonly have manes.\",\"draft_uniform_v1\"\n\"tiger\",\"A large solitary predatory cat with dark stripes on its coat.\",\"draft_uniform_v1\"\n\"wolf\",\"A wild social canine that commonly hunts in packs.\",\"draft_uniform_v1\"\n\"bear\",\"A large sturdy mammal with thick fur and strong claws.\",\"draft_uniform_v1\"\n\"giraffe\",\"A tall African mammal with a long neck and long legs.\",\"draft_uniform_v1\"\n\"zebra\",\"An African hoofed mammal with a black and white striped coat.\",\"draft_uniform_v1\"\n\"rabbit\",\"A small long-eared mammal with powerful hind legs that often burrows.\",\"draft_uniform_v1\"\n\"hare\",\"A long-eared mammal with long hind legs that typically rests above ground.\",\"draft_uniform_v1\"\n\"dolphin\",\"An aquatic mammal with a streamlined body and echolocation abilities.\",\"draft_uniform_v1\"\n\"jellyfish\",\"A marine animal with a gelatinous bell and usually tentacles.\",\"draft_uniform_v1\"\n\"octopus\",\"A marine mollusk with a soft body and eight arms.\",\"draft_uniform_v1\"\n\"squid\",\"A marine mollusk with an elongated body, eight arms and two tentacles.\",\"draft_uniform_v1\"\n\"shark\",\"A cartilaginous fish with gills and a streamlined predatory body.\",\"draft_uniform_v1\"\n\"starfish\",\"A marine echinoderm with arms radiating from a central disc.\",\"draft_uniform_v1\"\n\"sea urchin\",\"A marine echinoderm with a rounded body covered in movable spines.\",\"draft_uniform_v1\"\n\"ant\",\"A social insect with a narrow waist that typically lives in colonies.\",\"draft_uniform_v1\"\n\"dragonfly\",\"A flying insect with large eyes and two pairs of elongated wings.\",\"draft_uniform_v1\"\n\"butterfly\",\"A flying insect with broad scaled wings and typically clubbed antennae.\",\"draft_uniform_v1\"\n\"moth\",\"A scaled-wing insect often active at night with varied antenna shapes.\",\"draft_uniform_v1\"\n\"beetle\",\"An insect whose hardened front wings cover its delicate hind wings.\",\"draft_uniform_v1\"\n\"piano\",\"A keyboard instrument whose keys trigger hammers that strike strings.\",\"draft_uniform_v1\"\n\"organ\",\"A keyboard instrument producing sustained tones through pipes or electronic mechanisms.\",\"draft_uniform_v1\"\n\"violin\",\"A small bowed string instrument typically held beneath the chin.\",\"draft_uniform_v1\"\n\"cello\",\"A large bowed string instrument played upright between the knees.\",\"draft_uniform_v1\"\n\"trumpet\",\"A brass wind instrument with valves and a flared bell.\",\"draft_uniform_v1\"\n\"trombone\",\"A brass wind instrument typically using a slide to change pitch.\",\"draft_uniform_v1\"\n\"flute\",\"A wind instrument sounded by directing air across an opening.\",\"draft_uniform_v1\"\n\"clarinet\",\"A woodwind instrument with a single reed and cylindrical bore.\",\"draft_uniform_v1\"\n\"drum\",\"A percussion instrument sounded by striking a surface or stretched membrane.\",\"draft_uniform_v1\"\n\"tambourine\",\"A handheld percussion instrument with a frame and small metal jingles.\",\"draft_uniform_v1\"\n\"apple\",\"A rounded edible fruit with a firm flesh and central seed core.\",\"draft_uniform_v1\"\n\"pear\",\"An edible fruit typically narrow near its stem and broader below.\",\"draft_uniform_v1\"\n\"banana\",\"An elongated edible fruit with a peel and soft flesh.\",\"draft_uniform_v1\"\n\"pineapple\",\"A tropical fruit with a rough patterned rind and juicy flesh.\",\"draft_uniform_v1\"\n\"strawberry\",\"A soft red fruit with small seedlike structures on its surface.\",\"draft_uniform_v1\"\n\"raspberry\",\"A soft edible fruit composed of many small clustered drupelets.\",\"draft_uniform_v1\"\n\"lemon\",\"A typically yellow citrus fruit with sour acidic juice.\",\"draft_uniform_v1\"\n\"lime\",\"A typically green citrus fruit with a small size and acidic juice.\",\"draft_uniform_v1\"\n\"coconut\",\"A tropical fruit with a fibrous husk, hard shell and edible interior.\",\"draft_uniform_v1\"\n\"chair\",\"A seat for one person typically supported by legs and a backrest.\",\"draft_uniform_v1\"\n\"stool\",\"A seat for one person typically without a backrest or armrests.\",\"draft_uniform_v1\"\n\"sofa\",\"An upholstered seat designed to accommodate multiple people.\",\"draft_uniform_v1\"\n\"armchair\",\"An upholstered seat for one person with supports for the arms.\",\"draft_uniform_v1\"\n\"bed\",\"A piece of furniture providing a horizontal surface for sleeping.\",\"draft_uniform_v1\"\n\"desk\",\"A piece of furniture providing a work surface for writing or computing.\",\"draft_uniform_v1\"\n\"table\",\"A piece of furniture with a flat top supported above the floor.\",\"draft_uniform_v1\"\n\"bookshelf\",\"A piece of furniture with shelves used to hold books.\",\"draft_uniform_v1\"\n\"cabinet\",\"A piece of furniture with enclosed compartments for storage.\",\"draft_uniform_v1\"\n\"hammer\",\"A hand tool with a weighted head used to deliver blows.\",\"draft_uniform_v1\"\n\"mallet\",\"A striking tool with a broad head often made of wood or rubber.\",\"draft_uniform_v1\"\n\"screwdriver\",\"A hand tool used to turn screws.\",\"draft_uniform_v1\"\n\"wrench\",\"A hand tool used to grip and turn nuts or bolts.\",\"draft_uniform_v1\"\n\"scissors\",\"A cutting tool with two blades pivoting against each other.\",\"draft_uniform_v1\"\n\"pliers\",\"A handheld tool with pivoting jaws for gripping or bending objects.\",\"draft_uniform_v1\"\n\"power drill\",\"A powered tool that rotates a bit to bore holes or drive fasteners.\",\"draft_uniform_v1\"\n\"electric saw\",\"A powered cutting tool that moves a toothed blade through material.\",\"draft_uniform_v1\"\n\"rose\",\"A flowering shrub often bearing prickles and layered fragrant blossoms.\",\"draft_uniform_v1\"\n\"tulip\",\"A bulbous plant with upright stems and typically cup-shaped flowers.\",\"draft_uniform_v1\"\n\"cactus\",\"A plant adapted to dry conditions often with succulent stems and spines.\",\"draft_uniform_v1\"\n\"aloe\",\"A succulent plant typically forming rosettes of thick fleshy leaves.\",\"draft_uniform_v1\"\n\"sunflower\",\"A flowering plant with a tall stem and large composite flower heads.\",\"draft_uniform_v1\"\n\"daisy\",\"A flowering plant typically bearing white ray florets around a yellow center.\",\"draft_uniform_v1\"\n\"bamboo\",\"A woody grass with jointed stems that are often hollow.\",\"draft_uniform_v1\"\n\"reed\",\"A tall slender grasslike plant commonly growing near water.\",\"draft_uniform_v1\"\n\"oak\",\"A tree or shrub that produces acorns and often has lobed leaves.\",\"draft_uniform_v1\"\n\"maple\",\"A tree or shrub often bearing lobed leaves and winged seeds.\",\"draft_uniform_v1\"\n\"rain\",\"Precipitation consisting of liquid water droplets falling from clouds.\",\"draft_uniform_v1\"\n\"snow\",\"Precipitation consisting of ice crystals often grouped into flakes.\",\"draft_uniform_v1\"\n\"hail\",\"Precipitation consisting of solid balls or irregular lumps of ice.\",\"draft_uniform_v1\"\n\"fog\",\"A suspension of tiny water droplets near the ground that reduces visibility.\",\"draft_uniform_v1\"\n\"smoke\",\"Airborne particles and gases produced by burning material.\",\"draft_uniform_v1\"\n\"lightning\",\"A sudden atmospheric electrical discharge producing a bright flash.\",\"draft_uniform_v1\"\n\"thunder\",\"The sound produced by rapid air expansion around an atmospheric electrical discharge.\",\"draft_uniform_v1\"\n\"earthquake\",\"A sudden shaking of the ground caused by energy released within Earth.\",\"draft_uniform_v1\"\n\"volcanic eruption\",\"An event in which magma, gases or ash escape from a volcanic vent.\",\"draft_uniform_v1\"\n"
raw['draft']=DRAFT_CSV.encode()
hashes={k:digest(v) for k,v in raw.items()}
read=lambda k: pd.read_csv(io.BytesIO(raw[k]))
pairs=read('pairs'); scores=read('scores'); draft=read('draft')
keys=['candidate_id','category','target_x','comparator_y']
assert len(pairs)==50 and pairs.candidate_id.is_unique
assert scores.candidate_id.is_unique and len(scores)==50
joined=pairs.merge(scores[keys+['L22_auc','L23_auc','L24_auc','p_shuffle_plus_one','fdr_q','screen_pass']],on=keys,how='left',validate='one_to_one',indicator=True)
assert joined['_merge'].eq('both').all()
joined=joined.drop(columns='_merge')
assert draft.concept.is_unique and set(draft.concept)==set(pairs.target_x)|set(pairs.comparator_y)
assert json.loads(raw['cue_config'])['revision']==CFG['revision']
assert json.loads(raw['old_config'])['per_class']==8
joined['historical_pass']=(joined.L24_auc>=CFG['auc_min'])&(joined.fdr_q<=CFG['q_max'])
assert joined[['L22_auc','L23_auc','L24_auc','p_shuffle_plus_one','fdr_q']].notna().all().all()
assert (joined.historical_pass==joined.screen_pass.astype(str).str.lower().eq('true')).all()
print('Pairs:',len(joined),'Unique concepts:',len(draft),'Historical gate:',joined.historical_pass.sum())
display(joined.groupby('category').historical_pass.agg(['count','sum']))
save_df('concept_pair_candidates.csv',pairs)
save_df('concept_definitions.csv',draft)
save_df('probe_evaluation_historical.csv',scores)
save('config.json',json.dumps(CFG,ensure_ascii=False,indent=2))


In [ ]:
# A1. Distance measurement, fixed revision. Resume from identical-input checkpoint.
fingerprint=digest(json.dumps({'config':CFG,'inputs':hashes},sort_keys=True).encode())
ck_name='checkpoint_distances.json'
ck=get_remote(ck_name)
if ck is None and (ROOT/ck_name).exists(): ck=(ROOT/ck_name).read_bytes()
if ck is not None:
    state=json.loads(ck); assert state['fingerprint']==fingerprint,'Checkpoint input mismatch'
    dist=pd.DataFrame(state['rows'])
    print('Resumed',len(dist),'distance rows')
else:
    model=SentenceTransformer(CFG['embedding'],revision=CFG['revision'],device='cpu')
    def embed_distance(a,b):
        texts=list(dict.fromkeys(list(a)+list(b)))
        e=model.encode(texts,batch_size=32,normalize_embeddings=True,show_progress_bar=True)
        mapping=dict(zip(texts,e))
        return np.array([float(1-np.clip(np.dot(mapping[x],mapping[y]),-1,1)) for x,y in zip(a,b)])
    def measure(frame,source):
        frame=frame.copy()
        frame['definition_source']=source
        frame['distance_xy']=embed_distance(frame.definition_x,frame.definition_y)
        return frame
    mapping=draft.set_index('concept').definition
    full=pairs[keys].copy()
    full['definition_x']=full.target_x.map(mapping); full['definition_y']=full.comparator_y.map(mapping)
    existing_cue=read('existing_cue').set_index('concept').definition
    cue_ids=['C01','C27','C35','C23','C37']
    cue=pairs[pairs.candidate_id.isin(cue_ids)][keys].copy()
    cue['definition_x']=cue.target_x.map(existing_cue); cue['definition_y']=cue.comparator_y.map(existing_cue)
    pre=pairs[keys].merge(read('existing_pre'),on=['candidate_id','target_x','comparator_y'],validate='one_to_one')
    for f in (full,cue,pre): assert f[['definition_x','definition_y']].notna().all().all()
    dist=pd.concat([measure(full,'uniform_draft_50'),measure(cue,'existing_cue_v1'),measure(pre,'existing_pre_main')],ignore_index=True)
    save(ck_name,json.dumps({'fingerprint':fingerprint,'rows':dist.to_dict('records')},ensure_ascii=False,sort_keys=True),True)
assert len(dist[dist.definition_source=='uniform_draft_50'])==50
assert dist.distance_xy.between(0,2).all()
save_df('xy_distance.csv',dist)
display(dist.groupby('definition_source').distance_xy.agg(['count','min','median','max']))


In [ ]:
# A2. Pair-level analyses; do not pool definition sets.
metrics=dist.merge(joined,on=keys,validate='many_to_one')
main=metrics[metrics.definition_source=='uniform_draft_50'].copy()
main['distance_band']=pd.qcut(main.distance_xy,5,duplicates='drop').astype(str)
def corr(x,y,method):
    x=np.asarray(x); y=np.asarray(y)
    ok=np.isfinite(x)&np.isfinite(y); x=x[ok]; y=y[ok]
    if len(x)<3 or np.ptp(x)==0 or np.ptp(y)==0: return len(x),np.nan,np.nan
    result=(pearsonr if method=='Pearson' else spearmanr)(x,y)
    return len(x),float(result.statistic),float(result.pvalue)
rows=[]
for source,g in metrics.groupby('definition_source'):
    for layer in (22,23,24):
        for method in ('Pearson','Spearman'):
            n,r,p=corr(g.distance_xy,g[f'L{layer}_auc'],method)
            rows.append(dict(source=source,layer=layer,method=method,n=n,r=r,p_nominal=p))
correlations=pd.DataFrame(rows)
# Category-demeaned L24 association; sensitivity only, no p-value claim.
within=main.copy()
within['distance_centered']=within.distance_xy-within.groupby('category').distance_xy.transform('mean')
within['auc_centered']=within.L24_auc-within.groupby('category').L24_auc.transform('mean')
n,within_r,_=corr(within.distance_centered,within.auc_centered,'Pearson')
# Leave-one-category-out diagnostic: range of estimates, not an independent-sample CI.
loo=[]
for cat in sorted(main.category.unique()):
    sub=main[main.category!=cat]
    for method in ('Pearson','Spearman'):
        n,r,p=corr(sub.distance_xy,sub.L24_auc,method)
        loo.append(dict(excluded_category=cat,method=method,n=n,r=r))
# Definition sensitivity: same pair, two independent definition sources, no silent replacement.
sensitivity=metrics.pivot(index='candidate_id',columns='definition_source',values='distance_xy').reset_index()
for name in ('existing_cue_v1','existing_pre_main'):
    sensitivity[name+'_minus_draft']=sensitivity[name]-sensitivity.uniform_draft_50
category=main.groupby('category').agg(n=('candidate_id','size'),distance_mean=('distance_xy','mean'),distance_median=('distance_xy','median'),auc_mean=('L24_auc','mean'),auc_median=('L24_auc','median'),historical_pass_rate=('historical_pass','mean')).reset_index()
bands=main.groupby('distance_band',sort=False).agg(n=('candidate_id','size'),distance_min=('distance_xy','min'),distance_max=('distance_xy','max'),auc_mean=('L24_auc','mean'),auc_median=('L24_auc','median'),historical_pass_rate=('historical_pass','mean')).reset_index().sort_values('distance_min')
save_df('pair_evaluation_all.csv',metrics)
save_df('pair_evaluation_50.csv',main)
save_df('distance_auc_correlations.csv',correlations)
save_df('category_summary.csv',category)
save_df('distance_band_summary.csv',bands)
save_df('leave_one_category_out.csv',pd.DataFrame(loo))
save_df('definition_sensitivity.csv',sensitivity)
display(correlations); display(bands); display(main[keys+['distance_xy','L24_auc','fdr_q','historical_pass']].sort_values('distance_xy'))
print('Category-demeaned Pearson r:',within_r)


In [ ]:
# A3. Visualizations, plus five-pair v4 exploratory linkage (not causal).
def finish(name):
    plt.tight_layout(); plt.savefig(ROOT/name,dpi=160,bbox_inches='tight'); plt.show(); plt.close()
plt.figure(figsize=(7,4)); plt.hist(main.distance_xy,bins=12); plt.xlabel('X/Y cosine distance'); plt.ylabel('Pairs'); finish('distance_histogram.png')
plt.figure(figsize=(8,5))
for cat,g in main.groupby('category'):
    plt.scatter(g.distance_xy,g.L24_auc,label=cat,alpha=.8)
plt.axhline(.8,linestyle='--'); plt.xlabel('X/Y cosine distance'); plt.ylabel('Historical L24 ROC-AUC'); plt.legend(fontsize=7,bbox_to_anchor=(1.02,1),loc='upper left'); finish('distance_auc.png')
plt.figure(figsize=(10,5)); cats=sorted(main.category.unique())
plt.boxplot([main.loc[main.category==c,'distance_xy'] for c in cats],labels=cats); plt.xticks(rotation=45,ha='right'); plt.ylabel('X/Y cosine distance'); finish('distance_by_category.png')
plt.figure(figsize=(8,4)); plt.bar(range(len(bands)),bands.auc_mean); plt.xticks(range(len(bands)),bands.distance_band,rotation=30,ha='right'); plt.ylabel('Mean historical L24 AUC'); plt.xlabel('Measured quintiles'); plt.ylim(0,1); finish('auc_by_distance_band.png')
plt.figure(figsize=(8,4))
for l in (22,23,24): plt.scatter(main.distance_xy,main[f'L{l}_auc'],label=f'L{l}',alpha=.5)
plt.legend(); plt.xlabel('X/Y distance'); plt.ylabel('Historical AUC'); finish('layer_auc.png')
v4=read('v4').query("method=='joint'").rename(columns={'pair':'candidate_id','x':'target_x','y':'comparator_y'})
v4=v4.merge(metrics.query("definition_source=='existing_cue_v1'")[['candidate_id','target_x','comparator_y','distance_xy']],on=['candidate_id','target_x','comparator_y'],validate='one_to_one')
save_df('v4_joint_five_pairs_exploratory.csv',v4); display(v4[['target_x','comparator_y','distance_xy','n','rho','spread','full_target_met']])


In [ ]:
# A4. Report and durable outputs. Existing files are never overwritten.
p=correlations[(correlations.source=='uniform_draft_50')&(correlations.layer==24)]
lines=['# Experiment A — exploratory analysis','',
 f'Historical 50 pairs, {len(draft)} concepts; historical gate {main.historical_pass.sum()}/50.',
 'Uniform definitions are newly drafted, not independently audited. Existing-definition analyses are separate sensitivity sets.',
 'L22–24 only; L1–21 and layerwise p/q are unavailable. Historical L24 p/q use 499 permutations and old 8/8 texts; not v0.97 confirmation.',
 '', '## L24 distance association']
for _,row in p.iterrows(): lines.append(f"- {row.method}: r={row.r:.4f}, nominal p={row.p_nominal:.4g}, n={row.n}")
lines += [f'Category-demeaned Pearson r={within_r:.4f}.', '',
 'Nominal p values are exploratory: categories, shared concepts and bounded/tied AUC violate simple independence assumptions. Inspect leave-one-category-out and definition sensitivity. No causal or formal adoption claim.',
 'The 5-pair v4 linkage uses original definitions and incomplete selections; it is descriptive only.',
 'No suppression-output data or effect direction was used. No generation API calls. Experiment B/C/D not executed.']
save('analysis_summary.md','\n'.join(lines)+'\n')
manifest={'status':'complete_exploratory','fingerprint':fingerprint,'source_paths':paths,'inputs_sha256':hashes,
 'embedding_revision':CFG['revision'],'versions':{k:im.version(k) for k in ['sentence-transformers','transformers','torch','numpy','pandas','scipy','matplotlib']},
 'python':platform.python_version(),'seed':CFG['seed'],'api_calls':0,'n_pairs':len(main),
 'files':{f.name:digest(f.read_bytes()) for f in sorted(ROOT.iterdir()) if f.is_file() and f.name!='manifest.json'}}
save('manifest.json',json.dumps(manifest,indent=2,ensure_ascii=False))
if SAVE_GITHUB:
    # Manifest last; no conflicting writes or overwrite. If rerun after completion, use a fresh run_id.
    for f in sorted(ROOT.iterdir()):
        if f.is_file() and f.name!='manifest.json': put_remote(f.name,f.read_bytes())
    put_remote('manifest.json',(ROOT/'manifest.json').read_bytes())
import shutil
archive=shutil.make_archive(str(ROOT),'zip',ROOT)
print((ROOT/'analysis_summary.md').read_text())
print('ZIP:',archive)
from google.colab import files
files.download(archive)
